# Sanad سند — question answering over Saudi law

## Part 4: training on how people actually ask

Parts 2 and 3 found the same pattern twice. Training on ArabLegalEval questions, which an LLM wrote while reading the passage and which reuse its wording, helps on more questions of that kind but not on the 79 questions real people asked:

- the fine-tuned retriever got worse on human questions (Recall@1 63.3% → 58.2%);
- the fine-tuned ALLaM said "not found" for 44% of human questions whose answer was in front of it.

Here the training questions are first rewritten the way a non-lawyer would ask them (Qwen2.5-7B-Instruct, which sees the question but never the passage). Then:

1. the retriever is fine-tuned on the same questions in original wording, in rewritten wording, and both;
2. ALLaM is fine-tuned again on the rewritten questions, with fewer "not found" examples (12% instead of 25%).

Caveat stated up front: this change was motivated by the Najiz results, so Najiz is no longer a fully untouched test set. There is no other human-written test set for Saudi law; this is reported, not hidden.

Code: https://github.com/readaloush/sanad

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers rank_bm25 peft bitsandbytes accelerate
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Rewrite training questions

In [ ]:
!python -m sanad.rewrite --model Qwen/Qwen2.5-7B-Instruct --n 2400 --data data --out results/rewrites.json 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## 2. Retriever: same questions, original wording vs rewritten

In [ ]:
!python -m sanad.finetune_retriever --base intfloat/multilingual-e5-base --rewrites results/rewrites.json --recipes original-subset,rewrites,original+rewrites --epochs 2 --data data --out results/rw 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## 3. ALLaM, fine-tuned on rewritten questions with fewer 'not found' examples

In [ ]:
!python -m sanad.sft --model ALLaM-AI/ALLaM-7B-Instruct-preview --rewrites results/rewrites.json --p-unanswerable 0.12 --data data --out results/lora2 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!python -m sanad.answer_eval --models ALLaM-AI/ALLaM-7B-Instruct-preview --adapter results/lora2 --tag _lora2 --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## Keep the results

In [ ]:
!mkdir -p /kaggle/working/out && cp results/rewrites.json results/answers_lora2.json results/answer_samples_lora2.json /kaggle/working/out/ && cp results/rw/retriever.json /kaggle/working/out/retriever_rewrites.json && cp -r results/lora2 /kaggle/working/out/lora2
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out /kaggle/working/out/lora2